[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%20-%20NLP/11_Pretrained_Embedding.ipynb)


# Embeddings preentrenados — buscar en una lista con una pregunta

En **09** y **10** entrenaste Word2Vec con *tu* texto. Aquí usamos un modelo que **Google ya entrenó**: mandas frases por la API y recibes vectores listos.

**Ejercicio de hoy:** tenemos una **lista de 10 párrafos** — como entradas de un FAQ de telefonía (facturación, roaming, soporte, etc.). Algunos están en **español** y otros en **inglés** a propósito. Llega una **pregunta** del usuario. ¿Cuál párrafo es el más parecido en *significado*?

No es búsqueda por palabra exacta (como Ctrl+F). El embedding entiende que *"¿por qué no tengo señal en el extranjero?"* puede acercarse a un párrafo en inglés sobre *roaming* aunque no compartan las mismas palabras.

Pasos:

1. Conectar con la **Gemini API**.
2. Ver qué devuelve el modelo para **una frase**.
3. Revisar la **lista de 10 párrafos** + escribir una **pregunta**.
4. Calcular similitud y decir **cuál gana**.


## Setup — instalar librerías

- `google-genai` — cliente de la Gemini API.
- `python-dotenv` — leer la clave desde `.env`.
- `numpy` — comparar vectores.


In [ ]:
%pip install -q google-genai python-dotenv numpy

In [ ]:
import os
import textwrap
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from google import genai

BASE_DIR = Path(".")

## 1 — ¿Qué cambia respecto a Word2Vec?

| | Word2Vec (notebooks 09–10) | Embedding preentrenado (este cuaderno) |
|---|---|---|
| **Quién entrena** | Tú, con gensim/Keras | Google (millones de textos) |
| **Unidad típica** | Una **palabra** | Una **frase** o párrafo completo |
| **Dónde corre** | Tu laptop / Colab | Servidor de Google (API) |
| **Costo** | Gratis (solo tu tiempo) | Tier gratuito con límites; luego de pago |
| **Privacidad** | Todo local | El texto sale a internet |

La idea del vector es la misma: una lista de números que resume el **significado**. Si dos frases dicen cosas parecidas, sus vectores deberían quedar **cerca** (similitud coseno alta).


## 2 — Clave API en `.env` (Google AI Studio)

1. Entra a [Google AI Studio → API keys](https://aistudio.google.com/apikey) y crea una clave.
2. En la carpeta `Modulo 4 - NLP/`, crea o edita el archivo `.env` (está en `.gitignore`, no se sube a GitHub):

```bash
GOOGLE_API_KEY=tu_clave_aqui
```

También aceptamos el nombre `GEMINI_API_KEY` si ya lo tenías así.

Usaremos el modelo **`gemini-embedding-001`**: solo texto, multilingüe (incluye español). Documentación: [Embeddings | Gemini API](https://ai.google.dev/gemini-api/docs/embeddings).

> **Colab:** en el menú *Secrets* puedes guardar `GOOGLE_API_KEY` y leerla con `os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")` si no usas `.env`.


In [ ]:
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
# Local / Jupyter: lee la clave desde .env en esta carpeta
# load_dotenv(BASE_DIR / ".env", override=True)
# API_KEY = (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY") or "").strip()

In [ ]:
cliente = genai.Client(api_key=API_KEY)
MODELO_EMBEDDING = "gemini-embedding-001"

print(f"Cliente listo. Modelo: {MODELO_EMBEDDING}")

**Solo Colab** — si la celda de arriba falla por falta de `.env`, usa esta variante: descomenta todo el bloque y **salta** `load_dotenv`.


In [ ]:
# from google.colab import userdata
#
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# API_KEY = os.environ["GOOGLE_API_KEY"].strip()
#
# cliente = genai.Client(api_key=API_KEY)
# MODELO_EMBEDDING = "gemini-embedding-001"
# print(f"Cliente listo (Colab). Modelo: {MODELO_EMBEDDING}")


## 3 — Una frase muy sencilla

Mandamos **un solo texto** al método `embed_content`. La API devuelve un objeto con la lista de números (el embedding).

Piensa en el vector como una **huella numérica** de la frase: no es legible para humanos, pero sirve para comparar textos.


In [ ]:
frase = "El gato duerme en el sofá."

respuesta = cliente.models.embed_content(
    model=MODELO_EMBEDDING,
    contents=frase,
)

# El SDK devuelve una lista de embeddings; con un solo texto, tomamos el primero.
vector = np.array(respuesta.embeddings[0].values, dtype=np.float32)

print(f"Frase: {frase}")
print(f"Dimensiones del vector: {vector.shape[0]}")
print(f"Primeros 8 valores: {vector[:8]}")
print(f"Norma L2 (longitud del vector): {np.linalg.norm(vector):.4f}")

El modelo devuelve una lista larga de números (3072 por defecto). No hace falta leer cada cifra: lo usaremos para **comparar** textos entre sí.


## 4 — Funciones para comparar textos

Dos piezas: convertir texto → vector, y medir qué tan parecidos son dos vectores (**similitud coseno**: 1 = muy parecido, 0 = nada que ver).


In [ ]:
def embed_textos(textos):
    """Devuelve matriz (n_textos, n_dims) con los embeddings de Gemini."""
    if isinstance(textos, str):
        textos = [textos]
    resp = cliente.models.embed_content(
        model=MODELO_EMBEDDING,
        contents=textos,
    )
    return np.array([e.values for e in resp.embeddings], dtype=np.float32)


def similitud_coseno(a, b):
    """1.0 = misma dirección; 0 = ortogonales; valores negativos = opuestos."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## 5 — Lista de FAQ (telefonía) + una pregunta

Simulamos preguntas frecuentes de una **compañía de telefonía**: facturación, planes, roaming, soporte técnico, portabilidad, etc.

Cada elemento es un **párrafo de dos líneas** (como un artículo del reglamento o del centro de ayuda). Algunos están en **inglés** para ver si el modelo conecta significado aunque la pregunta venga en español.

Cambia `pregunta` en la celda siguiente y vuelve a ejecutar desde ahí.

In [ ]:
FAQ = [
    # 1 — Facturación (ES)
    "Si tu recibo mensual incluye cargos que no reconoces, revisa primero el desglose en la app Mi Cuenta "
    "y confirma si hubo consumo de datos fuera de plan o servicios de terceros activados por SMS. "
    "Para disputar un cargo, abre un ticket antes del día 10 posterior a la emisión de la factura.",

    # 2 — Roaming (EN)
    "If your mobile line shows No Service after traveling abroad, verify that data roaming is enabled "
    "in your device settings and that your plan includes international coverage. "
    "Contact support within 48 hours to restore your line without penalty if the issue persists.",

    # 3 — Cambio de plan (ES)
    "Para subir o bajar tu plan de datos, entra a Mi Cuenta con tu número registrado y elige la opción "
    "Cambiar plan en el menú de servicios. Los cambios aplican al inicio del siguiente ciclo de facturación, "
    "salvo que contrates un paquete adicional de datos, que se activa de inmediato.",

    # 4 — Sin señal / red (EN)
    "When calls drop frequently or you see only one bar indoors, restart your phone and confirm that "
    "VoLTE and automatic network selection are turned on. If the problem continues in the same location "
    "for more than 72 hours, request a network quality review from technical support.",

    # 5 — Portabilidad (ES)
    "La portabilidad numérica permite conservar tu número al cambiarte de compañía sin costo, siempre "
    "que no tengas adeudos pendientes ni contrato de permanencia vigente. Solicítala en la tienda "
    "de la nueva operadora con una identificación oficial; el trámite tarda entre 24 y 72 horas hábiles.",

    # 6 — Formas de pago (ES)
    "Puedes pagar tu factura en OXXO, transferencia SPEI, tarjeta de crédito o débito desde la app, "
    "y en ventanilla bancaria con tu referencia de 16 dígitos. El pago se refleja en un máximo de "
    "48 horas; después de esa fecha pueden aplicarse recargos por mora según tu contrato.",

    # 7 — Garantía del equipo (EN)
    "Devices purchased from our stores include a 12-month warranty against manufacturing defects, "
    "but not damage from liquids, falls, or unauthorized repairs. To start a warranty claim, "
    "bring the phone, the original invoice, and the IMEI number to any authorized service center.",

    # 8 — Fraude / SIM swap (ES)
    "Si recibes llamadas extrañas pidiendo tu NIP o códigos de verificación, no los compartas: "
    "puede tratarse de un intento de fraude o clonación de SIM. Reporta el incidente de inmediato "
    "al *611 o al chat de seguridad para bloquear tu línea y emitir una SIM nueva.",

    # 9 — Internet en casa (ES)
    "La instalación de fibra óptica incluye visita técnica, módem y configuración de la red Wi-Fi "
    "en tu domicilio; el técnico verificará que haya cobertura en tu colonia antes de perforar. "
    "Si ya tienes servicio y la velocidad es menor a la contratada, realiza una prueba por cable LAN.",

    # 10 — Horarios de atención (EN)
    "Customer service by phone is available Monday through Saturday from 8:00 a.m. to 8:00 p.m., "
    "and Sundays from 9:00 a.m. to 2:00 p.m. local time. For faster help, use the in-app chat "
    "or visit a retail store with your account number and a valid ID.",
]

print(f"Entradas en la lista: {len(FAQ)}")
for i, texto in enumerate(FAQ, start=1):
    print(f"\n--- [{i}] ---")
    print(textwrap.fill(texto, width=88))

In [ ]:
pregunta = "¿Por qué no tengo señal desde que aterricé en otro país?"

In [ ]:
print("Pregunta del usuario:")
print(f"  → {pregunta}")

## 6 — ¿Cuál párrafo de la lista gana?

Una llamada a la API: embedding de la pregunta + embedding de cada párrafo. **Mayor similitud coseno = más parecido en significado.**

In [ ]:
def imprimir_parrafo(texto, indent="   ", ancho=88):
    """Envuelve el texto para que no se corte feo en pantalla."""
    return textwrap.fill(texto, width=ancho, initial_indent=indent, subsequent_indent=indent)


def buscar_en_lista(pregunta, textos):
    """Devuelve lista ordenada: (índice, párrafo, similitud) de mayor a menor."""
    todos = [pregunta] + list(textos)
    vectores = embed_textos(todos)
    vec_pregunta = vectores[0]
    vec_entradas = vectores[1:]

    resultados = []
    for i, (texto, vec_entrada) in enumerate(zip(textos, vec_entradas)):
        score = similitud_coseno(vec_pregunta, vec_entrada)
        resultados.append((i + 1, texto, score))

    resultados.sort(key=lambda x: x[2], reverse=True)
    return resultados


ranking = buscar_en_lista(pregunta, FAQ)

print(f"Pregunta: {pregunta}\n")
print("Ranking (de más parecido a menos):")
print("=" * 88)

for pos, (num, texto, score) in enumerate(ranking, start=1):
    print(f"\n{pos:2d}. [#{num}]  similitud = {score:.4f}")
    print(imprimir_parrafo(texto))

ganador = ranking[0]
print("\n" + "=" * 88)
print(f"MEJOR COINCIDENCIA: #{ganador[0]}  (similitud = {ganador[2]:.4f})")
print("=" * 88)
print(imprimir_parrafo(ganador[1], indent=""))

## 7 — Prueba con otra pregunta

Cambia `pregunta` en la celda de arriba y vuelve a correr las celdas **5** y **6**. Ideas:

- `"Me cobraron algo raro en el recibo del mes"` → debería ir a **#1** (facturación)
- `"¿Cómo me cambio de compañía y me quedo con mi número?"` → **#5** (portabilidad)
- `"My phone screen broke after I dropped it, is it covered?"` → **#7** (garantía, en inglés)
- `"¿Hasta qué hora puedo marcar al call center?"` → **#10** (horarios, en inglés)

**Cierre.** Misma lógica que un chatbot de soporte: la pregunta entra, se compara contra una lista de respuestas guardadas, gana la más parecida. El modelo multilingüe ayuda cuando la pregunta y el párrafo no están en el mismo idioma.